# 11 - Reinforcement Learning

## Frozen Lake Game 

The Frozen Lake environment is a popular testbed in reinforcement learning developed by OpenAI Gym. This environment challenges an agent to navigate a grid-like frozen lake from a starting point to a goal, avoiding falling through holes.

![Example Image](https://www.gymlibrary.dev/_images/frozen_lake.gif)


### Environment Layout

- **Grid**: The environment is presented as a grid, typically 4x4 or 8x8, where each cell can represent either safe ice or a dangerous hole.
- **States**: Each cell in the grid is a distinct state an agent can occupy. The state number increases from top left to bottom right.
- **Start and Goal**:
  - **Start**: The agent starts from the top-left corner of the grid.
  - **Goal**: The objective is to reach the bottom-right corner of the grid.

### Symbols

- **S (Start)**: Safe starting position for the agent.
- **F (Frozen)**: Safe positions where the agent can step.
- **H (Hole)**: Holes that the agent should avoid as stepping into a hole ends the episode with a loss.
- **G (Goal)**: The target position which the agent aims to reach for a successful outcome.

### Actions

The agent can perform four possible actions:
1. **Left**
2. **Down**
3. **Right**
4. **Up**

These actions determine the movement direction of the agent on the grid.

### Dynamics

- **Deterministic vs. Stochastic**: The environment can be either:
  - **Deterministic**: Actions lead predictably to a specific adjoining state.
  - **Stochastic (Is Slippery)**: Actions may not always lead to the intended next state due to the slippery nature of the ice, mimicking real-life uncertainty and difficulty.

### Objectives and Rewards

- **Objective**: Navigate from start (S) to goal (G) safely without falling into any holes.
- **Rewards**:
  - **0**: For each step taken that does not result in reaching the goal or falling into a hole.
  - **1**: Upon successfully reaching the goal.
  - **Termination**: The episode ends when the agent falls into a hole, reaches the goal or reaches the max number of steps (truncation).

### Use in Reinforcement Learning

The Frozen Lake environment is used to train agents using various reinforcement learning algorithms, including but not limited to Q-Learning and SARSA. These algorithms help the agent learn optimal paths from start to goal while minimizing the risk of falling into holes.

### Challenges

- **Exploration vs. Exploitation**: Balancing the need to explore the environment to learn about dangerous areas (holes) while exploiting known safe paths to reach the goal efficiently.
- **Learning from Sparse Rewards**: Learning effective strategies in an environment where rewards are only received at the end of the episode.

The Frozen Lake game provides a simple yet challenging platform for developing and testing reinforcement learning algorithms, making it a staple in the AI and machine learning communities. 

In this assignment we will implement and test  **Model free q-learning** and **DeepQNetworks**.


# Setup

In [ ]:

!pip install gym
!pip install torch
!pip install matplotlib
!pip install numpy
!pip install gymnasium


In [ ]:


import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random
import torch
from torch import nn
import torch.nn.functional as F
import seaborn as sns
from gymnasium.envs.toy_text.frozen_lake import generate_random_map
import os
from IPython import display
from time import sleep


In [ ]:
# Seed everything for reproducible results
seed = 2024
def reset_seed(the_seed = 2024):
    np.random.seed(the_seed)
    np.random.default_rng(the_seed)
    os.environ['PYTHONHASHSEED'] = str(the_seed)
    torch.manual_seed(the_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(the_seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [ ]:
# Function to display the game env as an image
def show_state(env, ep):
    plt.figure(figsize=(4, 4))
    plt.title('Episode ' +str(ep), fontsize=16, fontweight='bold', color='red', loc='center')
    plt.imshow(env.render())
    plt.axis('off')  # Hide axes
    
    display.display(plt.gcf())
    sleep(0.1)
    display.clear_output(wait=True)
    plt.close()

In [ ]:
def qtable_directions_map(qtable, map_size):
    """Get the best learned action & map it to arrows."""
    qtable_val_max = qtable.max(axis=1).reshape(map_size, map_size)
    qtable_best_action = np.argmax(qtable, axis=1).reshape(map_size, map_size)
    directions = {0: "←", 1: "↓", 2: "→", 3: "↑"}
    qtable_directions = np.empty(qtable_best_action.flatten().shape, dtype=str)
    eps = np.finfo(float).eps  # Minimum float number on the machine
    for idx, val in enumerate(qtable_best_action.flatten()):
        if qtable_val_max.flatten()[idx] > eps:
            # Assign an arrow only if a minimal Q-value has been learned as best action
            # otherwise since 0 is a direction, it also gets mapped on the tiles where
            # it didn't actually learn anything
            qtable_directions[idx] = directions[val]
    qtable_directions = qtable_directions.reshape(map_size, map_size)
    return qtable_val_max, qtable_directions

def plot_q_values_map(qtable, map_size):
    """Plot the last frame of the simulation and the policy learned."""
    qtable_val_max, qtable_directions = qtable_directions_map(qtable, map_size)

    # Plot the last frame
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 5))
    
    # Plot the policy
    sns.heatmap(
        qtable_val_max,
        annot=qtable_directions,
        fmt="",
        ax=ax,
        cmap=sns.color_palette("Blues", as_cmap=True),
        linewidths=0.7,
        linecolor="black",
        xticklabels=[],
        yticklabels=[],
        annot_kws={"fontsize": "xx-large"},
    ).set(title="Learned Q-values\nArrows represent best action")
    for _, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_linewidth(0.7)
        spine.set_color("black")
    img_title = f"frozenlake_q_values_{map_size}x{map_size}.png"
    plt.show()

# Model-Free Q-Learning Algorithm

Q-learning is an off-policy reinforcement learning algorithm that seeks to find the best action to take given the current state. It's considered off-policy because the Q-learning function learns from actions that are outside the current policy, like taking random actions.

We are going to implement Q-Learning with Exploration and Exploitation. The following algorithm outlines the process of training an agent to navigate an environment (e.g., a grid world) where the goal is to reach a destination without falling into traps or holes.


## Algorithm Overview

1. **Initialize the Q-values (Q(s, a)) to zero for all state-action pairs**:
   - The Q-table represents the expected future rewards for action `a` in state `s`.

2. **For each episode (or trial):**
   - Initialize the state `s`.
   - **For each step of the episode:**
     - Draw a random number and check if < ε. If if < ε Choose next action randomly (Exploration). If not, choose an action `a` in the current world state `s` based on current Q-value estimates (Exploitation).
     - Take the action, and observe the reward `r` and the new state `s'`.
     - Update the Q-value for the state-action pair using the formula:
       ```
       Q(s, a) = Q(s, a) + α * [r + γ * max(Q(s', a')) - Q(s, a)]
       ```
       Where:
       - `α` is the learning rate.
       - `γ` is the discount factor.
       - `r` is the reward received after taking action `a` in state `s`.
       - `max(Q(s', a'))` is the estimate of optimal future value.
     - Update state `s` to `s'`.
   - Reduce ε (the exploration rate) gradually.

3. **Repeat** for each episode.

## Parameters

- **α (Learning Rate):** How much we accept the new value vs the old value.
- **γ (Discount Factor):** How much importance we give to future rewards.
- **ε (Exploration rate):** The probability of choosing a random action at each step.

## Benefits of Q-Learning

- **Off-policy:** It can learn from actions that are outside the current policy, hence learning optimal policy even if actions are random.
- **Convergence:** Given enough time and under certain conditions, Q-learning will converge to the optimal action-value function, Q*(s, a).

## Parameterisation of ε 

- It is crucial to balance the exploration (trying new actions) and exploitation (using known rewards) to ensure that the Q-learning algorithm works effectively.



### Based on the above mention, implement model free Q Learning. Implement steps: 1. up to 6.

In [ ]:
def model_free_q_learning(episodes, is_training=True, render=False, map_size=4 ,_is_slippery = False, train_params = None, q_table = None):

    env = gym.make('FrozenLake-v1',desc=generate_random_map(size=map_size, p=0.8, seed=seed) ,
                   is_slippery=_is_slippery, render_mode='rgb_array' if render else None)

    if(is_training):
        # 1. create the q table
        
        # 2. unpack train_params
        
        pass
    else:
        if q_table is None:
            f = open('frozen_lake'+map_size+'.pkl', 'rb')
            q_table = pickle.load(f)
            f.close()
    
    rewards_per_episode = np.zeros(episodes)


    for i in range(episodes):
        state = env.reset()[0]  # states
        terminated = False      # True when fall in hole or reached goal
        truncated = False       # True when actions > 200

        while(not terminated and not truncated):
            if True: # 3. Exploration phase
                action = env.action_space.sample() # actions: 0=left,1=down,2=right,3=up

            else:
                # 4. Exploitation phase
                pass 

            # perform action
            new_state,reward,terminated,truncated,_ = env.step(action)

            if render:
                show_state(env ,i)
            
            if is_training:
                # 5. Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
                pass

            state = new_state

        if is_training:
            # 6. Update epsilon. Should decrease over time
            
            
            # change Learning rate when epsilon gets to 0
            if(epsilon==0):
                learning_rate_a = 0.0001

        if reward == 1:
            rewards_per_episode[i] = 1

    env.close()

    sum_rewards = np.zeros(episodes)
    for t in range(episodes):
        sum_rewards[t] = np.sum(rewards_per_episode[max(0, t-100):(t+1)])
    plt.plot(sum_rewards)
    plt.savefig("frozen_lake"+ str(map_size)+".png")
    
    if is_training:
        f = open("frozen_lake"+ str(map_size)+".pkl","wb")
        pickle.dump(q_table, f)
        f.close()
    return q_table, env


# Setup of game env + Training call

In [ ]:
# env params
episodes = 5_000
is_training = True
show_map = False
map_conf = 4
is_slippery = True
seed = 2024
reset_seed(seed)
# training params
train_params={}
train_params["learning_rate_a"] = 0.9 # alpha or learning rate
train_params["discount_factor_gamma"] = 0.9 # gamma or discount rate. Near 0: more weight/reward placed on immediate state. Near 1: more on future state.
train_params["epsilon"]  = 1         # 1 = 100% random actions
train_params["epsilon_decay_rate"] = 0.0001        # epsilon decay rate. 1/0.0001 = 10,000
#
q_table, env = model_free_q_learning(episodes,is_training,show_map,map_conf,is_slippery,train_params=train_params)


    

In [ ]:
# Let us see the q-table
print(q_table) # show values of the final q table

# plot the decision per cell
plot_q_values_map(q_table, map_conf)

In [ ]:
# Test mode!
episodes = 10
is_training = False
show_map = True
#

q_table = model_free_q_learning(episodes,is_training,show_map,map_conf,is_slippery,q_table=q_table)


# Deep Q Networks 

# Deep Q-Networks (DQN) Algorithm

Deep Q-Networks (DQN) integrate classical Q-learning with deep neural networks to handle complex environments with high-dimensional state spaces. DQN uses two key neural network architectures: the policy network and the target network.

## Key Components

- **Policy Network**: This neural network approximates the Q-function. It predicts the Q-values for all possible actions in a given state.
- **Target Network**: This is a copy of the policy network that is held constant to stabilize training. Its weights are updated less frequently to provide consistent targets during temporal difference learning.

## DQN Algorithm Steps

1. **Initialize the Policy Network**:
   - Initialize the policy network with random weights.

2. **Copy the Policy Network to create the Target Network**:
   - Initialize the target network with the same weights as the policy network.

3. **Collect Experiences**:
   - For each episode, initialize the state.
   - For each step in the episode:
     - Select an action using the policy network and an ε-greedy policy.
     - Execute the action in the environment.
     - Observe the new state and reward.
     - Store the transition (state, action, reward, next state) in a replay buffer (memory).

4. **Sample a Random Batch from the Replay Buffer (Memory)**:
   - Randomly sample a batch of transitions from the replay buffer to break the correlation between consecutive updates.

5. **Compute the Target Value**:
   - For each sampled transition, calculate the target Q-value using the Bellman equation:
     ```
     y_i = r + γ * max_a' Q_target(s', a')
     ```
     where `r` is the reward, `γ` is the discount factor, and `Q_target(s', a')` is the Q-value predicted by the target network for the next state.

6. **Update the Policy Network**:
   - Calculate the loss between the predicted Q-values from the policy network and the target Q-values:
     ```
     Loss = (y_i - Q_policy(s, a))^2
     ```
   - Perform a gradient descent step to minimize this loss. Note: it is the MSELoss.

7. **Periodically Update the Target Network**:
   - Every few steps, update the weights of the target network to match the weights of the policy network. This frequency is a hyperparameter that needs tuning.

8. **Decay ε Gradually**:
   - Reduce ε over time to decrease the rate of exploration and increase exploitation.

## Considerations for Implementation

- **Experience Replay**: This technique is crucial in DQN to stabilize training by using a buffer that stores past experiences. It allows the network to learn from a diverse set of experiences and avoid local minima.
- **Fixed Q-Targets**: Using a separate target network helps in stabilizing the learning process by providing fixed targets during updates.
- **ε-Greedy Strategy**: This strategy balances exploration and exploitation by choosing random actions with a probability of ε and the best-known action with a probability of 1-ε.

DQN can effectively learn policies directly from high-dimensional sensory inputs using end-to-end training. 

This methodology has been demonstrated to perform well on a variety of challenging tasks.

### Implement steps 1. to 5. of the algorithm in FrozenLakeDQL class.


In [ ]:
# Define the DQN model (simple example!)
class DQN(nn.Module):
    def __init__(self, inputs, h1_nodes, outputs):
        super().__init__()

        # Define network layers
        self.fc1 = nn.Linear(inputs, h1_nodes)   # first fully connected layer, with inputs being the number of available states
        self.out = nn.Linear(h1_nodes, outputs) # ouptut layer with output nodes mapped into the available actions

    def forward(self, x):
        x = F.relu(self.fc1(x)) # Apply rectified linear unit (ReLU) activation
        x = self.out(x)         # Calculate output
        return x

# Define memory for Experience Replay
class ReplayMemory():
    def __init__(self, maxlen):
        self.memory = deque([], maxlen=maxlen)
    
    def append(self, transition):
        self.memory.append(transition)

    def sample(self, sample_size):
        return random.sample(self.memory, sample_size)

    def __len__(self):
        return len(self.memory)

In [ ]:
# FrozeLake Deep Q-Learning
class FrozenLakeDQL():
    
    ACTIONS = ['L','D','R','U']     # for printing 0,1,2,3 => L(eft),D(own),R(ight),U(p)

    def __init__(self, hyperparam, env_settings):
        
        self.hyperparam = hyperparam
        self.env_settings = env_settings
        self.env = gym.make('FrozenLake-v1',desc=generate_random_map(size=env_settings["map_size"], p=env_settings["p_frozen"], seed=env_settings["seed"]) ,
                            is_slippery=env_settings["is_slippery"], render_mode='rgb_array' if env_settings["render"] else None)
        self.loss_fn = nn.MSELoss()          # NN Loss function. MSE=Mean Squared Error can be swapped to something else.
        self.optimizer = None                # NN Optimizer. Initialize later.

    def train(self, episodes, render=False):
        # Create FrozenLake instance
        
        num_states = self.env.observation_space.n
        num_actions = self.env.action_space.n
        
        epsilon = 1 # 1 = 100% random actions
        memory = ReplayMemory(self.hyperparam["replay_memory_size"])

        ######## 1. Create policy and target network. Number of nodes in the hidden layer can be adjusted.
        

        # Make the target and policy networks the same (copy weights/biases from one network to the other)
        ### example:
        #target_dqn.load_state_dict(policy_dqn.state_dict())

        print('Policy (random, before training):')
        self.print_dqn(policy_dqn)

        # Policy network optimizer. 
        if self.hyperparam["TOptimizer"] == "Adam": 
            self.optimizer = torch.optim.Adam(policy_dqn.parameters(), lr=self.hyperparam["learning_rate_a"])
        else:
            print("Not implemented!")
        
        # List to keep track of rewards collected per episode. Initialize list to 0's.
        rewards_per_episode = np.zeros(episodes)

        # List to keep track of epsilon decay
        epsilon_history = []

        # Track number of steps taken. Used for syncing policy => target network.
        step_count=0
            
        for i in range(episodes):
            state = self.env.reset()[0]  # Initialize to state 0
            terminated = False      # True when agent falls in hole or reached goal
            truncated = False       # True when agent takes more than 200 actions    

            # Agent navigates map until it falls into hole/reaches goal (terminated), or has taken 200 actions (truncated).
            while(not terminated and not truncated):

                # Select action based on epsilon-greedy
                if random.random() < epsilon:
                    # select random action
                    action = self.env.action_space.sample() # actions: 0=left,1=down,2=right,3=up
                else:
                    ############ 2. select best action            
                    with torch.no_grad():
                        # action = ...  # from which network?

                # Execute action
                new_state, reward, terminated, truncated, _ = self.env.step(action)

                ######### 3. Save experience into memory
                

                # Move to the next state
                state = new_state

                # Increment step counter
                step_count+=1

            # Keep track of the rewards collected per episode.
            if reward == 1:
                rewards_per_episode[i] = 1

            # Check if enough experience has been collected and if at least 1 reward has been collected
            if len(memory) > self.hyperparam["mini_batch_size"] and np.sum(rewards_per_episode)>0:
                
                mini_batch = memory.sample(self.hyperparam["mini_batch_size"])  # to train the model, sample a mini batch of experiences from memory
                # go!
                self.optimize(mini_batch, policy_dqn, target_dqn)        

                # Decay epsilon
                epsilon = max(epsilon - 1/episodes, 0)
                epsilon_history.append(epsilon)

                # Copy policy network to target network after a certain number of steps
                if step_count > self.hyperparam["network_sync_rate"]:
                    target_dqn.load_state_dict(policy_dqn.state_dict())
                    step_count=0

        # Close environment
        self.env.close()

        # Save policy
        torch.save(policy_dqn.state_dict(), "frozen_lake_dql.pt")

        # Create new graph 
        plt.figure(1)

        # Plot average rewards (Y-axis) vs episodes (X-axis)
        sum_rewards = np.zeros(episodes)
        for x in range(episodes):
            sum_rewards[x] = np.sum(rewards_per_episode[max(0, x-100):(x+1)])
        plt.subplot(121) # plot on a 1 row x 2 col grid, at cell 1
        plt.plot(sum_rewards)
        
        # Plot epsilon decay (Y-axis) vs episodes (X-axis)
        plt.subplot(122) # plot on a 1 row x 2 col grid, at cell 2
        plt.plot(epsilon_history)
        
        # Save plots
        plt.savefig('frozen_lake_dql.png')
        self.print_dqn(policy_dqn)
        

    # Optimize policy network
    def optimize(self, mini_batch, policy_dqn, target_dqn):

        # Get number of input nodes
        num_states = policy_dqn.fc1.in_features

        current_q_list = []
        target_q_list = []

        for state, action, new_state, reward, terminated in mini_batch:

            if terminated: 
                # Agent either reached goal (reward=1) or fell into hole (reward=0)
                # When in a terminated state, target q value should be set to the reward.
                target = torch.FloatTensor([reward])
            else:
                # Calculate target q value 
                with torch.no_grad():
                    target = torch.FloatTensor(
                        reward + self.hyperparam["discount_factor_g"] * target_dqn(self.state_to_dqn_input(new_state, num_states)).max()
                    )

            on_hot_enc = self.state_to_dqn_input(state, num_states)
            
            # Get the current set of Q values
            current_q = policy_dqn(on_hot_enc)
            current_q_list.append(current_q)

            # Get the target set of Q values
            target_q = target_dqn(on_hot_enc) 
            
            # Adjust the specific action to the target that was just calculated
            target_q[action] = target
            target_q_list.append(target_q)
                
        # Compute loss for the whole minibatch
        loss = self.loss_fn(torch.stack(current_q_list), torch.stack(target_q_list))
        print("loss", loss)
        # Optimize the model
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    '''
    Converts an state (int) to a tensor representation. It is a simple one hot encoding.
    For example, the FrozenLake 4x4 map has 4x4=16 states numbered from 0 to 15. 

    Parameters: state=1, num_states=16
    Return: tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
    '''
    def state_to_dqn_input(self, state:int, num_states:int)->torch.Tensor:
        input_tensor = torch.zeros(num_states)
        input_tensor[state] = 1
        return input_tensor

    # Run the FrozeLake environment with the learned policy
    def test(self, episodes, render=True):
        # Create FrozenLake instance
        
        if render:
            self.env = gym.make('FrozenLake-v1',desc=generate_random_map(size=env_settings["map_size"], p=env_settings["p_frozen"], seed=env_settings["seed"]) ,
                            is_slippery=env_settings["is_slippery"], render_mode='rgb_array')
        
        num_states = self.env.observation_space.n
        num_actions = self.env.action_space.n

        # Load learned policy
        policy_dqn = DQN(inputs=num_states, h1_nodes=num_states, outputs=num_actions) 
        policy_dqn.load_state_dict(torch.load("frozen_lake_dql.pt"))
        policy_dqn.eval()    # switch model to evaluation mode

        print('Policy (trained):')
        self.print_dqn(policy_dqn)

        for i in range(episodes):
            state = self.env.reset()[0]  # Initialize to state 0
            terminated = False      # True when agent falls in hole or reached goal
            truncated = False       # True when agent takes more than 200 actions            

            # Agent navigates map until it falls into a hole (terminated), reaches goal (terminated), or has taken 200 actions (truncated).
            while(not terminated and not truncated):  
                # Select best action   
                with torch.no_grad():
                    ######### 4. execute the policy_dqn and get next action
                    pass

                ######### 5. Execute action
                
                
                if render:
                    show_state(self.env, i)

        self.env.close()

    # Print DQN: state, best action, q values
    def print_dqn(self, dqn):
        # Get number of input nodes
        num_states = dqn.fc1.in_features

        # Loop each state and print policy to console
        for s in range(num_states):
            #  Format q values for printing
            q_values = ''
            for q in dqn(self.state_to_dqn_input(s, num_states)).tolist():
                q_values += "{:+.2f}".format(q)+' '  # Concatenate q values, format to 2 decimals
            q_values=q_values.rstrip()              # Remove space at the end

            # Map the best action to L D R U
            best_action = self.ACTIONS[dqn(self.state_to_dqn_input(s, num_states)).argmax()]

            # Print policy in the format of: state, action, q values
            # The printed layout matches the FrozenLake map.
            print(f'{s:02},{best_action},[{q_values}]', end=' ')         
            if (s+1)%4==0:
                print() # Print a newline every 4 states




Let us train de DQNetworks 

In [ ]:


# DQN Hyperparameters (adjustable)
hyperparam = {}
hyperparam["TOptimizer"] = "Adam"             # Select optimizer 
hyperparam["learning_rate_a"] = 0.001         # learning rate (alpha)
hyperparam["discount_factor_g"] = 0.9         # discount rate (gamma)    
hyperparam["network_sync_rate"] = 10          # number of steps the agent takes before syncing the policy and target network
hyperparam["replay_memory_size"]  = 1000      # size of replay memory
hyperparam["mini_batch_size"] = 32            # size of the training data set sampled from the replay memory

training_ep = 1000 # train episodes

# Frozen lake Env settings
env_settings = {}
env_settings["is_slippery"] = True
env_settings["map_size"] = 4
env_settings["seed"] = 2024
env_settings["p_frozen"] = 0.6
env_settings["render"] = False # for training
frozen_lake = FrozenLakeDQL(hyperparam=hyperparam, env_settings=env_settings)

frozen_lake.train(training_ep)


And test it. It makes sense to test more than one episode if the *is_slippery* (the stochastic version) is set to True

In [ ]:
test_ep = 5 # for is_slippery = True makes sense to test more than once
frozen_lake.test(test_ep)

### Explore! Try using diferent map sizes, parameters, DQN.

References:

-  https://gymnasium.farama.org/tutorials/training_agents/FrozenLake_tuto/
-  https://github.com/MehdiShahbazi/DQN-Frozenlake-Gymnasium/tree/master
-  https://karan-jakhar.medium.com/100-days-of-code-day-1-35afe174000e
- https://github.com/johnnycode8/gym_solutions/blob/main/README.md